#### Ray Tracing acceleration

### Objective Explore the running time issues of obtaining photo-realistic images and accelartion methods.

##### Q. How many Calculations are actually needed to render an image

1. We shoot rays through each pixel $\rightarrow$ # of Rays is dependant on the number pixels. 
2. For each ray we split into two more rays thus is max level = n $\rightarrow$ per pixel we'll have $2^{n-1} -1 $ rays.
3. So for the whole pixel image we'll have $n \cdot 2^{n-1} -1$

- We can conclude that the algorithm won't be so efficient so we try a different approach

#### Accelerate the Computation for each ray

- For each ray we try to find an intersection in the scene ```FindIntersection()```, to understand the factor of complication this add. 
- Suppose we have an image with $N, M \in \N \mid \ N•M$ pixels
- $L$ light sources 
- $K \in \N$ Levels of recursion

- The number of time that for each pixel we build a tree of $MN(2^K -1)$ rays.  
- We will then shoot at least $(L+1)$ Shadow rays leading to $$MN(2^K -1*(L+1))$$ number of calls to ```FindIntersection()```

#### Back to the ```FindIntersection()```

```java
Intersection FindIntersection (Ray ray, Scene scene)
{
    min_t = -nf
    min_primitive = NULL
    For each primitice in scene {
        t = Intersect(ray, primitive)
        if (t < min_t) {
            min_primitive = primitive
            min_t = t
        }
    }
    return Intersetion(min_t, min_primitive)
}
```


### Fast Reject (Accelaration Method)

- Assume we have an complex object int he scene with many triangles, which construct the the surface of the object.
- If We want to find the intersection with a ray and the  object, we'd need to go over all the triangle and to find the shortest point of intersection. 
    - For example: Assume your object is a simple closed box. The number of ray-triangle intersection calculations are needed to find the intersection of the object with a ray is  
    12 Since two trangles for a square plane and we'd need 6 planes for the box. 

- Fast reject will enclose each object in the secne by a cube, bounded by the $MAX: HIGHT, LENGTH, WIDTH$ per object in the scene.
- We will then only need to check if the ray insects a box than the complex object (of manyb triangles), by a constant factor (12 per object).
- Only once we intersect a perticular box, do we start investigating the object within.
<p align="center">
    <img src="images_U5/Screenshot 2025-07-02 at 10.54.22.png" height="400" width="400"/>
    <img src="images_U5/Screenshot 2025-07-02 at 10.53.38.png" height="400" width="400"/>
</p>

### Lets go even further (Bounding Volume Hierarchy):

- We can consider each feature of a perticular object and enclose them with a box.
- We can apply this to every object. 
- Categorize sectors of a scene with merging larger Boxes, based on the proximity of groups of objects in the scene and continue this process until we have a single box(cube).

- We then use this $bounding \ box \ hierarchy$ process to produce a tree (not necessarily binary). 
- Then shoot a ray from the centre of the camera and have two benificial cases: 
    1. If there was no intersection with the single box, we can move on to a new ray angle (saving a lot of computation)
    2. If it hit the first layer box then, go down the tree and check for further intersections with the children, continue with this until we reach a leaf (object in the scene)   
    or not in which case return to step 1, but with added information to potentially change the angle of the next ray by a smaller change (as we're likely to hit an object). 

|Scene View | Data Structure View |
|-----------|---------------------|
|<img src="images_U5/Screenshot 2025-07-02 at 10.55.01.png" height="400" width="400"/>|<img src="images_U5/Screenshot 2025-07-02 at 10.55.08.png" height="400" width="400"/>|
|<img src="images_U5/Screenshot 2025-07-02 at 10.55.18.png" height="400" width="400"/>|<img src="images_U5/Screenshot 2025-07-02 at 10.55.33.png" height="400" width="400"/>|
|<img src="images_U5/Screenshot 2025-07-02 at 10.57.03.png" height="400" width="400"/> |<img src="images_U5/Screenshot 2025-07-02 at 10.57.10.png" height="400" width="400"/>|
|<img src="images_U5/Screenshot 2025-07-02 at 10.57.22.png" height="400" width="400"/> |<img src="images_U5/Screenshot 2025-07-02 at 10.57.29.png" height="400" width="400"/>|
|<img src="images_U5/Screenshot 2025-07-02 at 10.58.29.png" height="400" width="400"/>|<img src="images_U5/Screenshot 2025-07-02 at 10.58.35.png" height="400" width="400"/>|

```cpp
FindIntersection (Ray, ray, Scene scene){
    //Recursion base: Intersect with an object
    if (isLeaf(Node)){
        t = findIntersection(ray, getObject(node));
        return t;
    }
    // Find intersection with bounding volumes of child node, assigning inf if not
    ... 
    // Sort intersections front to back (smallest to largest values)
    // insert into array BV_t[i]
    ...
    // Process Intersections
    // Checking for early termination
    min_t = infinity; 
    for i = 1; i < numChildren; i++ {
        if (min_t < BV_t[i]) break;
        t = FindIntersection(ray, getSortChild(i)); 
        if (t < min_t) { min_t = t; }
    }
    return min_t;
}
```